In [111]:
from __future__ import annotations
import json
import re
from pathlib import Path
import numpy as np
import pandas as pd
pd.set_option("display.max_columns", None)

In [112]:
ROOT = Path("/home/anastass/spoc-masked-attention/results/teacher-attention/iter_5000/lambda-scaling-test-kappa0p2")
OUT_DIR = Path("/home/anastass/spoc-masked-attention/results/teacher-attention/analysis/lambda-scaling-test-kappa0p2")
OUT_DIR.mkdir(parents=True, exist_ok=True)
AGG_PATH = OUT_DIR / "aggregated_lambda_sweep_all_rows.csv"

In [113]:
summary_files = sorted(ROOT.rglob("summary.csv"))
config_files = sorted(ROOT.rglob("config*.json"))

print("Number of summary.csv files:", len(summary_files))
print("Number of config json files:", len(config_files))

print("\nExample summary files:")
for p in summary_files[:5]:
    print(p)

print("\nExample config files:")
for p in config_files[:5]:
    print(p)

Number of summary.csv files: 20
Number of config json files: 200

Example summary files:
/home/anastass/spoc-masked-attention/results/teacher-attention/iter_5000/lambda-scaling-test-kappa0p2/maskrandom_r_10_rstar_10_sigstar_1_bstar_1_beta_1_d50_T5_lambda0p01_lr0p001_iter5000_pca10/maskrandom_r_10_rstar_10_sigstar_1_bstar_1_beta_1_d50_T5_lambda0p01_lr0p001_iter5000_pca10_seed_42_54047204/summary.csv
/home/anastass/spoc-masked-attention/results/teacher-attention/iter_5000/lambda-scaling-test-kappa0p2/maskrandom_r_10_rstar_10_sigstar_1_bstar_1_beta_1_d50_T5_lambda0p025_lr0p001_iter5000_pca10/maskrandom_r_10_rstar_10_sigstar_1_bstar_1_beta_1_d50_T5_lambda0p025_lr0p001_iter5000_pca10_seed_42_54047205/summary.csv
/home/anastass/spoc-masked-attention/results/teacher-attention/iter_5000/lambda-scaling-test-kappa0p2/maskrandom_r_10_rstar_10_sigstar_1_bstar_1_beta_1_d50_T5_lambda0p05_lr0p001_iter5000_pca10/maskrandom_r_10_rstar_10_sigstar_1_bstar_1_beta_1_d50_T5_lambda0p05_lr0p001_iter5000_pca10

In [114]:
def flatten_dict(d: dict, parent_key: str = "", sep: str = ".") -> dict:
    out = {}
    for k, v in d.items():
        key = f"{parent_key}{sep}{k}" if parent_key else str(k)
        if isinstance(v, dict):
            out.update(flatten_dict(v, key, sep=sep))
        else:
            out[key] = v
    return out

In [115]:
def load_json(path: Path) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

In [116]:
def infer_n_train(path: Path) -> int | None:
    text = str(path)
    m = re.search(r"ntrain[_-](\d+)", text)
    return int(m.group(1)) if m else None

In [117]:
def infer_seed(path: Path) -> int | None:
    text = str(path)
    m = re.search(r"seed[_-](\d+)", text)
    return int(m.group(1)) if m else None

In [118]:
def parse_config_folder_name(name: str) -> dict:
    """
    Parse metadata from folder names such as:
    maskrandom_r_25_rstar_25_sigstar_1_bstar_1_beta_1_d25_T5_lambda0p1_lr0p001_iter5000_pca25
    """
    def to_float(s: str | None):
        if s is None:
            return None
        return float(s.replace("p", "."))

    def grab(pattern: str):
        m = re.search(pattern, name)
        return m.group(1) if m else None

    return {
        "folder_config_name": name,
        "folder_mask_label": grab(r"^(mask[^_]+)"),
        "folder_r": int(grab(r"_r_(\d+)")) if grab(r"_r_(\d+)") else None,
        "folder_r_star": int(grab(r"_rstar_(\d+)")) if grab(r"_rstar_(\d+)") else None,
        "folder_d": int(grab(r"_d(\d+)")) if grab(r"_d(\d+)") else None,
        "folder_T": int(grab(r"_T(\d+)")) if grab(r"_T(\d+)") else None,
        "folder_lambda_reg": to_float(grab(r"_lambda([0-9p]+)")),
        "folder_lr": to_float(grab(r"_lr([0-9p]+)")),
        "folder_n_steps": int(grab(r"_iter(\d+)")) if grab(r"_iter(\d+)") else None,
        "folder_pca_n_components": int(grab(r"_pca(\d+)")) if grab(r"_pca(\d+)") else None,
    }

In [119]:
def get_top_config_folder(path: Path, root: Path = ROOT) -> Path:
    """
    For a path inside:
        ROOT / config_folder / job_folder / ntrain_folder / config.json

    return:
        ROOT / config_folder
    """
    rel = path.relative_to(root)
    return root / rel.parts[0]

In [120]:
rows = []

for config_path in config_files:
    try:
        config = load_json(config_path)
    except Exception as exc:
        print(f"[skip] could not read {config_path}: {exc}")
        continue

    flat = flatten_dict(config)

    top_config_folder = get_top_config_folder(config_path)
    run_folder = config_path.parent
    job_folder = run_folder.parent

    row = {
        "top_config_folder": str(top_config_folder),
        "top_config_folder_name": top_config_folder.name,
        "job_folder": str(job_folder),
        "run_folder": str(run_folder),
        "config_json": str(config_path),
        "inferred_n_train": infer_n_train(run_folder),
        "inferred_seed": infer_seed(run_folder),
        **parse_config_folder_name(top_config_folder.name),
        **flat,
    }

    rows.append(row)

config_df = pd.DataFrame(rows)

print(config_df.shape)
config_df.head()

(200, 55)


,top_config_folder,top_config_folder_name,job_folder,run_folder,config_json,inferred_n_train,inferred_seed,folder_config_name,folder_mask_label,folder_r,folder_r_star,folder_d,folder_T,folder_lambda_reg,folder_lr,folder_n_steps,folder_pca_n_components,experiment.save_root,experiment.run_name,experiment.seed,experiment.master_seed,experiment.run_seed,experiment.teacher_seed,experiment.train_data_seed,experiment.population_data_seed,experiment.student_init_seed,experiment.random_baseline_seed,data.data_model,data.T,data.d,data.mask_value,data.masking_strategy,data.masks_per_sample,teacher.init,teacher.r_star,teacher.beta_star,teacher.sigma_star,model.r,model.beta,model.normalize_sqrt_d,model.dtype,model.device,training.alpha,training.n_train,training.n_steps,training.learning_rate,training.lambda_reg,evaluation.n_population,evaluation.eval_every,evaluation.track_attention_error_during_training,evaluation.attention_metric_subset_size,evaluation.pca_n_components,evaluation.n_random_baselines,logging.use_wandb,logging.project
0,/home/anastass/spoc-masked-attention/results/t...,maskrandom_r_10_rstar_10_sigstar_1_bstar_1_bet...,/home/anastass/spoc-masked-attention/results/t...,/home/anastass/spoc-masked-attention/results/t...,/home/anastass/spoc-masked-attention/results/t...,10000,42,maskrandom_r_10_rstar_10_sigstar_1_bstar_1_bet...,maskrandom,10,10,50,5,0.01,0.001,5000,10,results/teacher-attention/iter_5000/lambda-sca...,None,35895890,42,35895890,42,35895891,10000042,20000042,30000042,teacher_attention,5,50,1.0,random,1,scaled_gaussian,10,1.0,1.0,10,1.0,True,float64,cpu,1.0,10000,5000,0.001,0.01,5000,25,True,512,10,10,False,spoc-masked-attention
1,/home/anastass/spoc-masked-attention/results/t...,maskrandom_r_10_rstar_10_sigstar_1_bstar_1_bet...,/home/anastass/spoc-masked-attention/results/t...,/home/anastass/spoc-masked-attention/results/t...,/home/anastass/spoc-masked-attention/results/t...,1000,42,maskrandom_r_10_rstar_10_sigstar_1_bstar_1_bet...,maskrandom,10,10,50,5,0.01,0.001,5000,10,results/teacher-attention/iter_5000/lambda-sca...,None,1658970725,42,1658970725,42,1658970726,10000042,20000042,30000042,teacher_attention,5,50,1.0,random,1,scaled_gaussian,10,1.0,1.0,10,1.0,True,float64,cpu,1.0,1000,5000,0.001,0.01,5000,25,True,512,10,10,False,spoc-masked-attention
2,/home/anastass/spoc-masked-attention/results/t...,maskrandom_r_10_rstar_10_sigstar_1_bstar_1_bet...,/home/anastass/spoc-masked-attention/results/t...,/home/anastass/spoc-masked-attention/results/t...,/home/anastass/spoc-masked-attention/results/t...,100,42,maskrandom_r_10_rstar_10_sigstar_1_bstar_1_bet...,maskrandom,10,10,50,5,0.01,0.001,5000,10,results/teacher-attention/iter_5000/lambda-sca...,None,933496318,42,933496318,42,933496319,10000042,20000042,30000042,teacher_attention,5,50,1.0,random,1,scaled_gaussian,10,1.0,1.0,10,1.0,True,float64,cpu,1.0,100,5000,0.001,0.01,5000,25,True,512,10,10,False,spoc-masked-attention
3,/home/anastass/spoc-masked-attention/results/t...,maskrandom_r_10_rstar_10_sigstar_1_bstar_1_bet...,/home/anastass/spoc-masked-attention/results/t...,/home/anastass/spoc-masked-attention/results/t...,/home/anastass/spoc-masked-attention/results/t...,20000,42,maskrandom_r_10_rstar_10_sigstar_1_bstar_1_bet...,maskrandom,10,10,50,5,0.01,0.001,5000,10,results/teacher-attention/iter_5000/lambda-sca...,None,3087836661,42,3087836661,42,3087836662,10000042,20000042,30000042,teacher_attention,5,50,1.0,random,1,scaled_gaussian,10,1.0,1.0,10,1.0,True,float64,cpu,1.0,20000,5000,0.001,0.01,5000,25,True,512,10,10,False,spoc-masked-attention
4,/home/anastass/spoc-masked-attention/results/t...,maskrandom_r_10_rstar_10_sigstar_1_bstar_1_bet...,/home/anastass/spoc-masked-attention/results/t...,/home/anastass/spoc-masked-attention/results/t...,/home/anastass/spoc-masked-attention/results/t...,2000,42,maskrandom_r_10_rstar_10_sigstar_1_bstar_1_bet...,maskrandom,10,10,50,5,0.01,0.001,5000,10,results/teacher-attention/iter_5000/lambda-sca...,None,4005546871,42,4005546871

In [121]:
def first_existing(df: pd.DataFrame, candidates: list[str]) -> str | None:
    for c in candidates:
        if c in df.columns:
            return c
    return None

In [122]:
standard_cols = {
    "d": ["data.d", "folder_d"],
    "T": ["data.T", "folder_T"],
    "r": ["model.r", "folder_r"],
    "r_star": ["teacher.r_star", "folder_r_star"],
    "beta": ["model.beta"],
    "beta_star": ["teacher.beta_star"],
    "sigma_star": ["teacher.sigma_star"],
    "lambda_reg": ["training.lambda_reg", "folder_lambda_reg"],
    "learning_rate": ["training.learning_rate", "folder_lr"],
    "n_steps": ["training.n_steps", "folder_n_steps"],
    "n_train": ["training.n_train", "inferred_n_train"],
    "seed": ["experiment.seed", "experiment.master_seed", "inferred_seed"],
    "masking_strategy": ["data.masking_strategy"],
    "masks_per_sample": ["data.masks_per_sample"],
    "pca_n_components": ["evaluation.pca_n_components", "folder_pca_n_components"],
}

for new_col, candidates in standard_cols.items():
    col = first_existing(config_df, candidates)
    if col is not None:
        config_df[new_col] = config_df[col]
    else:
        config_df[new_col] = np.nan

In [123]:
numeric_cols = [
    "d", "T", "r", "r_star", "beta", "beta_star", "sigma_star",
    "lambda_reg", "learning_rate", "n_steps", "n_train", "seed",
    "masks_per_sample", "pca_n_components",
]

for c in numeric_cols:
    if c in config_df.columns:
        config_df[c] = pd.to_numeric(config_df[c], errors="coerce")

In [124]:
config_df["kappa"] = config_df["r"] / config_df["d"]
config_df["kappa_star"] = config_df["r_star"] / config_df["d"]
config_df["alpha_linear"] = config_df["n_train"] / config_df["d"]
config_df["alpha_quadratic"] = config_df["n_train"] / (config_df["d"] ** 2)
config_df["clt_baseline"] = 1.0 / np.sqrt(config_df["d"])
config_df["theoretical_psd_baseline"] = config_df["kappa_star"] / (1.0 + config_df["kappa_star"])

In [125]:
summary_rows = []

for summary_path in summary_files:
    try:
        df = pd.read_csv(summary_path)
    except Exception as exc:
        print(f"[skip] could not read {summary_path}: {exc}")
        continue

    if df.empty:
        print(f"[skip] empty summary: {summary_path}")
        continue

    top_config_folder = get_top_config_folder(summary_path)
    df = df.copy()

    df["top_config_folder"] = str(top_config_folder)
    df["top_config_folder_name"] = top_config_folder.name
    df["summary_csv"] = str(summary_path)
    df["summary_parent"] = str(summary_path.parent)

    summary_rows.append(df)

summary_df = pd.concat(summary_rows, ignore_index=True) if summary_rows else pd.DataFrame()

print(summary_df.shape)
summary_df.head()

(200, 79)


,alpha,master_seed,run_seed,seed,teacher_seed,train_data_seed,population_data_seed,student_init_seed,n_train,n_population,teacher_init,r_star,beta_star,sigma_star,train_loss,population_risk,generalization_gap,ridge_train_loss,ridge_population_risk,ridge_generalization_gap,attention_vs_ridge_gap,attention_vs_ridge_relative_improvement,pca_train_loss,pca_population_risk,pca_generalization_gap,attention_vs_pca_gap,attention_vs_pca_relative_improvement,pca_n_components,cosine_S_S_star,random_baseline_cosine_S_S_star,relative_error_S_S_star,final_attention_level_error,runtime_seconds,runtime_per_step_seconds,initial_objective,final_objective,best_objective,objective_reduction,initial_train_loss_history,final_train_loss_history,best_train_loss_history,train_loss_reduction,weight_norm,W_star_norm,S_trace,S_top_eigenvalue,S_min_eigenvalue,S_R1,S_effective_rank,S_frobenius_norm,S_star_trace,S_star_top_eigenvalue,S_star_min_eigenvalue,S_star_R1,S_star_effective_rank,S_star_frobenius_norm,datetime,run_name,baseline_sqrt_d,centered_cosine_S_S_star,config_suffix,experiment_number,kappa,kappa_star,n_population_loss_terms,n_random_baselines,n_train_loss_terms,r,random_baseline_centered_cosine_S_S_star,random_baseline_centered_cosine_S_S_star_mean,random_baseline_centered_cosine_S_S_star_std,random_baseline_cosine_S_S_star_mean,random_baseline_cosine_S_S_star_std,random_baseline_seed,ridge_lambda,top_config_folder,top_config_folder_name,summary_csv,summary_parent
0,NaN,42,2187595824,2187595824,42,2187595825,10000042,20000042,25,5000,scaled_gaussian,10,1.0,1.0,0.079823,0.274401,0.194577,0.100166,0.243408,0.143242,0.030992,-0.127327,0.110503,0.352778,0.242274,-0.078377,0.222172,10,0.143344,0.173355,2.239208,0.034194,28.470930,0.005694,0.172632,0.092371,0.092371,0.080261,0.170840,0.079823,0.079823,0.091017,12.547145,9.226996,28.056269,12.325201,-2.143056e-15,0.439303,4.185166,15.250343,20.63219,3.675948,-9.498012e-16,0.178166,9.132642,7.086674,20260508-134912,ntrain_25_seed_42_20260508-134912,7.071068,0.041163,teacher_attention__init_scaled_gaussian__rstar...,NaN,0.2,0.2,5000,10,25,10,0.006251,0.006251,0.025813,0.173355,0.021688,30000042,0.01,/home/anastass/spoc-masked-attention/results/t...,maskrandom_r_10_rstar_10_sigstar_1_bstar_1_bet...,/home/anastass/spoc-masked-attention/results/t...,/home/anastass/spoc-masked-attention/results/t...
1,NaN,42,933496318,933496318,42,933496319,10000042,20000042,100,5000,scaled_gaussian,10,1.0,1.0,0.166968,0.226845,0.059876,0.186204,0.190695,0.004491,0.036149,-0.189566,0.277166,0.338490,0.061324,-0.111646,0.329834,10,0.196900,0.173355,2.772976,0.037536,32.962813,0.006593,0.261566,0.184618,0.184618,0.076948,0.259730,0.166968,0.166968,0.092762,17.649706,9.226996,39.465943,16.046846,-3.142543e-15,0.406600,5.098939,19.777282,20.63219,3.675948,-9.498012e-16,0.178166,9.132642,7.086674,20260508-134912,ntrain_100_seed_42_20260508-134912,7.071068,0.092312,teacher_attention__init_scaled_gaussian__rstar...,NaN,0.2,0.2,5000,10,100,10,0.006251,0.006251,0.025813,0.173355,0.021688,30000042,0.01,/home/anastass/spoc-masked-attention/results/t...,maskrandom_r_10_rstar_10_sigstar_1_bstar_1_bet...,/home/anastass/spoc-masked-attention/results/t...,/home/anastass/spoc-masked-attention/results/t...
2,NaN,42,3832603222,3832603222,42,3832603223,10000042,20000042,200,5000,scaled_gaussian,10,1.0,1.0,0.154177,0.201357,0.047180,0.155040,0.187722,0.032682,0.013635,-0.072634,0.260947,0.329866,0.068920,-0.128509,0.389578,10,0.189766,0.173355,1.972061,0.031615,36.799582,0.007360,0.217696,0.167653,0.167653,0.050044,0.215953,0.154177,0.154177,0.061776,13.475029,9.226996,30.131080,10.122363,-2.010467e-15,0.335944,6.154677,13.464963,20.63219,3.675948,-9.498012e-16,0.178166,9.132642,7.086674,20260508-134912,ntrain_200_seed_42_20260508-134912,7.071068,0.068790,teacher_attention__init_scaled_gaussian__rstar...,NaN,0.2,0.2,5000,10,200,10,0.006251,0.006251,0.025813,0.173355,0.021688,30000042,0.01,/home/anastass/spoc-masked-attention/results/t...,maskr

In [126]:
summary_df.columns.tolist()

['alpha',
 'master_seed',
 'run_seed',
 'seed',
 'teacher_seed',
 'train_data_seed',
 'population_data_seed',
 'student_init_seed',
 'n_train',
 'n_population',
 'teacher_init',
 'r_star',
 'beta_star',
 'sigma_star',
 'train_loss',
 'population_risk',
 'generalization_gap',
 'ridge_train_loss',
 'ridge_population_risk',
 'ridge_generalization_gap',
 'attention_vs_ridge_gap',
 'attention_vs_ridge_relative_improvement',
 'pca_train_loss',
 'pca_population_risk',
 'pca_generalization_gap',
 'attention_vs_pca_gap',
 'attention_vs_pca_relative_improvement',
 'pca_n_components',
 'cosine_S_S_star',
 'random_baseline_cosine_S_S_star',
 'relative_error_S_S_star',
 'final_attention_level_error',
 'runtime_seconds',
 'runtime_per_step_seconds',
 'initial_objective',
 'final_objective',
 'best_objective',
 'objective_reduction',
 'initial_train_loss_history',
 'final_train_loss_history',
 'best_train_loss_history',
 'train_loss_reduction',
 'weight_norm',
 'W_star_norm',
 'S_trace',
 'S_top_eige

In [127]:
summary_n_col = first_existing(summary_df, ["n_train", "ntrain", "training.n_train"])
summary_seed_col = first_existing(summary_df, ["seed", "experiment.seed", "experiment.master_seed"])

if summary_n_col is None:
    raise ValueError("Could not find n_train column in summary_df.")

summary_df["n_train"] = pd.to_numeric(summary_df[summary_n_col], errors="coerce")

if summary_seed_col is not None:
    summary_df["seed"] = pd.to_numeric(summary_df[summary_seed_col], errors="coerce")
else:
    summary_df["seed"] = np.nan

In [128]:
config_merge = config_df.copy()
config_merge["top_config_folder"] = config_merge["top_config_folder"].astype(str)

summary_merge = summary_df.copy()
summary_merge["top_config_folder"] = summary_merge["top_config_folder"].astype(str)

summary_has_seed = summary_merge["seed"].notna().any()
config_has_seed = config_merge["seed"].notna().any()

if summary_has_seed and config_has_seed:
    merge_keys = ["top_config_folder", "n_train", "seed"]
else:
    merge_keys = ["top_config_folder", "n_train"]

print("Merge keys:", merge_keys)

aggregated = summary_merge.merge(
    config_merge,
    on=merge_keys,
    how="left",
    suffixes=("", "_config"),
)

print(aggregated.shape)
aggregated.head()

Merge keys: ['top_config_folder', 'n_train', 'seed']
(200, 152)


,alpha,master_seed,run_seed,seed,teacher_seed,train_data_seed,population_data_seed,student_init_seed,n_train,n_population,teacher_init,r_star,beta_star,sigma_star,train_loss,population_risk,generalization_gap,ridge_train_loss,ridge_population_risk,ridge_generalization_gap,attention_vs_ridge_gap,attention_vs_ridge_relative_improvement,pca_train_loss,pca_population_risk,pca_generalization_gap,attention_vs_pca_gap,attention_vs_pca_relative_improvement,pca_n_components,cosine_S_S_star,random_baseline_cosine_S_S_star,relative_error_S_S_star,final_attention_level_error,runtime_seconds,runtime_per_step_seconds,initial_objective,final_objective,best_objective,objective_reduction,initial_train_loss_history,final_train_loss_history,best_train_loss_history,train_loss_reduction,weight_norm,W_star_norm,S_trace,S_top_eigenvalue,S_min_eigenvalue,S_R1,S_effective_rank,S_frobenius_norm,S_star_trace,S_star_top_eigenvalue,S_star_min_eigenvalue,S_star_R1,S_star_effective_rank,S_star_frobenius_norm,datetime,run_name,baseline_sqrt_d,centered_cosine_S_S_star,config_suffix,experiment_number,kappa,kappa_star,n_population_loss_terms,n_random_baselines,n_train_loss_terms,r,random_baseline_centered_cosine_S_S_star,random_baseline_centered_cosine_S_S_star_mean,random_baseline_centered_cosine_S_S_star_std,random_baseline_cosine_S_S_star_mean,random_baseline_cosine_S_S_star_std,random_baseline_seed,ridge_lambda,top_config_folder,top_config_folder_name,summary_csv,summary_parent,top_config_folder_name_config,job_folder,run_folder,config_json,inferred_n_train,inferred_seed,folder_config_name,folder_mask_label,folder_r,folder_r_star,folder_d,folder_T,folder_lambda_reg,folder_lr,folder_n_steps,folder_pca_n_components,experiment.save_root,experiment.run_name,experiment.seed,experiment.master_seed,experiment.run_seed,experiment.teacher_seed,experiment.train_data_seed,experiment.population_data_seed,experiment.student_init_seed,experiment.random_baseline_seed,data.data_model,data.T,data.d,data.mask_value,data.masking_strategy,data.masks_per_sample,teacher.init,teacher.r_star,teacher.beta_star,teacher.sigma_star,model.r,model.beta,model.normalize_sqrt_d,model.dtype,model.device,training.alpha,training.n_train,training.n_steps,training.learning_rate,training.lambda_reg,evaluation.n_population,evaluation.eval_every,evaluation.track_attention_error_during_training,evaluation.attention_metric_subset_size,evaluation.pca_n_components,evaluation.n_random_baselines,logging.use_wandb,logging.project,d,T,r_config,r_star_config,beta,beta_star_config,sigma_star_config,lambda_reg,learning_rate,n_steps,masking_strategy,masks_per_sample,pca_n_components_config,kappa_config,kappa_star_config,alpha_linear,alpha_quadratic,clt_baseline,theoretical_psd_baseline
0,NaN,42,2187595824,2187595824,42,2187595825,10000042,20000042,25,5000,scaled_gaussian,10,1.0,1.0,0.079823,0.274401,0.194577,0.100166,0.243408,0.143242,0.030992,-0.127327,0.110503,0.352778,0.242274,-0.078377,0.222172,10,0.143344,0.173355,2.239208,0.034194,28.470930,0.005694,0.172632,0.092371,0.092371,0.080261,0.170840,0.079823,0.079823,0.091017,12.547145,9.226996,28.056269,12.325201,-2.143056e-15,0.439303,4.185166,15.250343,20.63219,3.675948,-9.498012e-16,0.178166,9.132642,7.086674,20260508-134912,ntrain_25_seed_42_20260508-134912,7.071068,0.041163,teacher_attention__init_scaled_gaussian__rstar...,NaN,0.2,0.2,5000,10,25,10,0.006251,0.006251,0.025813,0.173355,0.021688,30000042,0.01,/home/anastass/spoc-masked-attention/results/t...,maskrandom_r_10_rstar_10_sigstar_1_bstar_1_bet...,/home/anastass/spoc-masked-attention/results/t...,/home/anastass/spoc-masked-attention/results/t...,maskrandom_r_10_rstar_10_sigstar_1_bstar_1_bet...,/home/anastass/spoc-masked-attention/results/t...,/home/anastass/spoc-masked-attention/results/t...,/home/anastass/spoc-masked-attention/results/t...,25,42,maskrandom_r_10_rstar_10_sigstar_1_bstar_1_bet...,maskrandom,10,10,50,5,0.01,0.001,5000,10,results/teacher-attention/iter_5000/lambda-sca...,None,21875

In [129]:
aggregated.to_csv(AGG_PATH, index=False)
print("Saved:", AGG_PATH)

Saved: /home/anastass/spoc-masked-attention/results/teacher-attention/analysis/lambda-scaling-test-kappa0p2/aggregated_lambda_sweep_all_rows.csv


In [130]:
important_cols = [
    "d", "r", "r_star", "kappa", "kappa_star",
    "lambda_reg", "learning_rate", "n_train",
    "alpha_linear", "alpha_quadratic",
    "masking_strategy", "masks_per_sample",
    "top_config_folder_name",
]

available = [c for c in important_cols if c in aggregated.columns]
aggregated[available].head()

,d,r,r_star,kappa,kappa_star,lambda_reg,learning_rate,n_train,alpha_linear,alpha_quadratic,masking_strategy,masks_per_sample,top_config_folder_name
0,50,10,10,0.2,0.2,0.01,0.001,25,0.5,0.01,random,1,maskrandom_r_10_rstar_10_sigstar_1_bstar_1_bet...
1,50,10,10,0.2,0.2,0.01,0.001,100,2.0,0.04,random,1,maskrandom_r_10_rstar_10_sigstar_1_bstar_1_bet...
2,50,10,10,0.2,0.2,0.01,0.001,200,4.0,0.08,random,1,maskrandom_r_10_rstar_10_sigstar_1_bstar_1_bet...
3,50,10,10,0.2,0.2,0.01,0.001,500,10.0,0.20,random,1,maskrandom_r_10_rstar_10_sigstar_1_bstar_1_bet...
4,50,10,10,0.2,0.2,0.01,0.001,1000,20.0,0.40,random,1,maskrandom_r_10_rstar_10_sigstar_1_bstar_1_bet...


In [131]:
cosine_candidates = [
    "cosine_S_S_star",
    "final_cosine_S_S_star",
    "teacher_cosine",
    "cosine_similarity",
]

cosine_col = None

for c in cosine_candidates:
    if c in aggregated.columns:
        cosine_col = c
        break

if cosine_col is None:
    possible = [
        c for c in aggregated.columns
        if "cosine" in c.lower()
        and "centered" not in c.lower()
        and "random" not in c.lower()
        and "baseline" not in c.lower()
    ]
    if possible:
        cosine_col = possible[0]

print("Using cosine column:", cosine_col)

Using cosine column: cosine_S_S_star


In [132]:
if cosine_col is None:
    raise ValueError("No learned non-centered cosine column found.")

aggregated["cosine"] = pd.to_numeric(aggregated[cosine_col], errors="coerce")

In [133]:
def summarize_lambda_group(g: pd.DataFrame, large_n_quantile: float = 0.7) -> pd.Series:
    g = g.dropna(subset=["n_train", "cosine"]).sort_values("n_train")

    n_threshold = g["n_train"].quantile(large_n_quantile)
    g_large = g[g["n_train"] >= n_threshold]

    final_row = g.loc[g["n_train"].idxmax()]

    out = {
        "n_points": len(g),
        "n_min": g["n_train"].min(),
        "n_max": g["n_train"].max(),
        "large_n_threshold": n_threshold,
        "cosine_mean_all": g["cosine"].mean(),
        "cosine_median_all": g["cosine"].median(),
        "cosine_max": g["cosine"].max(),
        "cosine_std_all": g["cosine"].std(ddof=0),
        "cosine_mean_large_n": g_large["cosine"].mean(),
        "cosine_median_large_n": g_large["cosine"].median(),
        "cosine_final": final_row["cosine"],
        "n_train_final": final_row["n_train"],
    }

    optional_cols = [
        "train_loss",
        "population_risk",
        "generalization_gap",
        "relative_error_S_S_star",
        "attention_level_error",
        "top_eigenvalue",
        "top_eigenvalue_S",
        "S_top_eigenvalue",
        "trace_S",
        "effective_rank",
    ]

    for c in optional_cols:
        if c in g.columns:
            values = pd.to_numeric(g[c], errors="coerce")
            if values.notna().any():
                out[f"{c}_mean_large_n"] = values.loc[g_large.index].mean()
                out[f"{c}_final"] = values.loc[final_row.name]

    return pd.Series(out)

In [136]:
lambda_summary = (
    aggregated
    .dropna(subset=["d", "lambda_reg", "n_train", "cosine"])
    .groupby(["d", "lambda_reg"])
    .apply(summarize_lambda_group)
    .reset_index()
    .sort_values(["d", "lambda_reg"])
    .reset_index(drop=True)
)

lambda_summary_path = OUT_DIR / "lambda_summary_by_d.csv"
lambda_summary.to_csv(lambda_summary_path, index=False)

lambda_summary

,d,lambda_reg,n_points,n_min,n_max,large_n_threshold,cosine_mean_all,cosine_median_all,cosine_max,cosine_std_all,cosine_mean_large_n,cosine_median_large_n,cosine_final,n_train_final,train_loss_mean_large_n,train_loss_final,population_risk_mean_large_n,population_risk_final,generalization_gap_mean_large_n,generalization_gap_final,relative_error_S_S_star_mean_large_n,relative_error_S_S_star_final,S_top_eigenvalue_mean_large_n,S_top_eigenvalue_final
0,25,0.010,10.0,25.0,40000.0,6500.0,0.605172,0.646862,0.846235,0.226748,0.841029,0.842522,0.846235,40000.0,0.222008,0.223890,0.226493,0.226539,0.004485,0.002649,0.558087,0.547102,3.546425e+00,3.521987e+00
1,25,0.025,10.0,25.0,40000.0,6500.0,0.692915,0.791998,0.891211,0.230599,0.884579,0.884156,0.891211,40000.0,0.229577,0.231441,0.234016,0.234096,0.004439,0.002655,0.625210,0.625156,1.756671e+00,1.710368e+00
2,25,0.050,10.0,25.0,40000.0,6500.0,0.685244,0.783328,0.836213,0.194044,0.831167,0.830717,0.836213,40000.0,0.239454,0.241302,0.243523,0.243684,0.004069,0.002383,0.865125,0.868467,7.039871e-01,6.655602e-01
3,25,0.100,10.0,25.0,40000.0,6500.0,0.363062,0.330293,0.458168,0.056776,0.327521,0.324781,0.324781,40000.0,0.247078,0.248659,0.250628,0.250631,0.003550,0.001972,0.999088,0.999126,1.544405e-02,1.493851e-02
4,25,0.250,10.0,25.0,40000.0,6500.0,0.253406,0.248638,0.326735,0.026654,0.249688,0.249651,0.249651,40000.0,0.247164,0.248741,0.250708,0.250708,0.003544,0.001967,1.000000,1.000000,7.095323e-06,6.735225e-06
5,50,0.010,10.0,25.0,40000.0,6500.0,0.455202,0.429485,0.801631,0.250573,0.763529,0.775942,0.801631,40000.0,0.191968,0.190410,0.192638,0.192584,0.000670,0.002174,0.718774,0.659044,3.962631e+00,3.991159e+00
6,50,0.025,10.0,25.0,40000.0,6500.0,0.551917,0.630940,0.880644,0.295163,0.869806,0.873857,0.880644,40000.0,0.198999,0.197386,0.199338,0.199344,0.000339,0.001958,0.545840,0.531313,2.596137e+00,2.665199e+00
7,50,0.050,10.0,25.0,40000.0,6500.0,0.548035,0.664982,0.804233,0.261523,0.801261,0.799800,0.804233,40000.0,0.209156,0.207423,0.209184,0.209105,0.000028,0.001682,0.774221,0.770356,1.537272e+00,1.590956e+00
8,50,0.100,10.0,25.0,40000.0,6500.0,0.307556,0.316623,0.409326,0.095423,0.383997,0.406419,0.406419,40000.0,0.221769,0.220128,0.221537,0.221535,-0.000231,0.001407,0.999862,0.999817,2.497917e-03,3.189139e-03
9,50,0.250,10.0,25.0,40000.0,6500.0,0.158661,0.169344,0.185593,0.028796,0.181896,0.182221,0.182221,40000.0,0.221779,0.220140,0.221547,0.221547,-0.000232,0.001407,1.000000,1.000000,3.700656e-11,3.654384e-11


In [135]:
score_col = "cosine_median_large_n"

idx = lambda_summary.groupby("d")[score_col].idxmax()

best_lambda_by_d = (
    lambda_summary
    .loc[idx]
    .sort_values("d")
    .reset_index(drop=True)
)

best_path = OUT_DIR / "best_lambda_by_d.csv"
best_lambda_by_d.to_csv(best_path, index=False)

best_lambda_by_d[
    [
        "d",
        "lambda_reg",
        score_col,
        "cosine_mean_large_n",
        "cosine_final",
        "cosine_max",
        "n_points",
        "n_min",
        "n_max",
    ]
]

,d,lambda_reg,cosine_median_large_n,cosine_mean_large_n,cosine_final,cosine_max,n_points,n_min,n_max
0,25,0.025,0.884156,0.884579,0.891211,0.891211,10.0,25.0,40000.0
1,50,0.025,0.873857,0.869806,0.880644,0.880644,10.0,25.0,40000.0
2,100,0.025,0.839325,0.822920,0.882213,0.882213,10.0,25.0,40000.0
3,200,0.050,0.671989,0.652140,0.753473,0.753473,10.0,25.0,40000.0
